In [ ]:
import pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'README.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / 'Data'
OUT  = ROOT / 'Code' / 'output'


# Figuras para la presentación de avances

Insumos: `{DATA}/processed/*.dta` y `{DATA}/raw/geo/ar_provincias.geojson`. Salidas: `{OUT}/figuras/av_*.png`. Las líneas de reforma van siempre etiquetadas con nombre y fecha de vigencia.

In [ ]:
import os, json, unicodedata
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
dat = str(DATA / "processed"); figd = os.path.join(str(OUT), "figuras")
os.makedirs(figd, exist_ok=True)
bal = pd.read_stata(os.path.join(dat, "balances_bancos.dta"))
pdp = pd.read_stata(os.path.join(dat, "prestamos_depositos_prov.dta"))
inc = pd.read_stata(os.path.join(dat, "inclusion_prov.dta"))
bancos = bal[bal.es_banco == 1].copy()
QR, PULL = pd.Timestamp("2021-11-01"), pd.Timestamp("2022-09-01")
QR_LBL, PULL_LBL = "QR interoperable\n29-nov-2021", "Transferencias pull\n30-sep-2022"
def norm(p):
    p = unicodedata.normalize("NFD", str(p).strip().upper())
    p = "".join(c for c in p if not unicodedata.combining(c))
    return "CABA" if ("CIUDAD" in p or "CAPITAL" in p) else p

## 1. El país en dos fotos: adopción provincial pre y post reformas

Adopción = personas con al menos una cuenta de pago como % de las personas con alguna cuenta. Misma escala de color en ambos paneles.

In [ ]:
def adop_at(q):
    x = inc[inc.ym == q].copy()
    return (100 * x.n_pers_min1_cta_pago / x.n_pers_min1_cta).set_axis(x.provincia).to_dict()
a_pre, a_post = adop_at("2021-09-01"), adop_at("2023-12-01")
gj = json.load(open(os.path.join(str(DATA / "raw"), "geo", "ar_provincias.geojson")))
vmin = min(min(a_pre.values()), min(a_post.values())); vmax = max(max(a_pre.values()), max(a_post.values()))
cmap = plt.cm.YlGnBu
fig, axes = plt.subplots(1, 2, figsize=(10, 6.5))
for ax, vals, title in [(axes[0], a_pre, "2021q3, previo a las reformas"), (axes[1], a_post, "2023q4, posterior a ambas")]:
    patches, colors = [], []
    for f in gj["features"]:
        prov = norm(f["properties"]["name"])
        geom = f["geometry"]
        polys = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
        for poly in polys:
            patches.append(MplPolygon(poly[0]))
            colors.append(vals.get(prov, np.nan))
    pc = PatchCollection(patches, cmap=cmap, edgecolor="white", linewidth=0.4)
    pc.set_array(np.array(colors)); pc.set_clim(vmin, vmax)
    ax.add_collection(pc); ax.set_xlim(-76, -52); ax.set_ylim(-56, -21)
    ax.set_aspect(1.15); ax.axis("off"); ax.set_title(title, fontsize=11)
cb = fig.colorbar(pc, ax=axes, shrink=0.7, pad=0.02)
cb.set_label("Personas con cuenta de pago, % de personas con alguna cuenta", fontsize=9)
cb.ax.tick_params(labelsize=8)
fig.savefig(os.path.join(figd, "av_mapa_adopcion.png"), dpi=170, bbox_inches="tight"); plt.close(fig)
print({k: round(v,1) for k,v in sorted(a_pre.items(), key=lambda kv: -kv[1])[:3]}, "...",
      {k: round(v,1) for k,v in sorted(a_post.items(), key=lambda kv: -kv[1])[:3]})

## 2. Validez de la exposición: pesos estables y adopción que crece pareja

In [ ]:
pre = pdp[pdp.ym.isin(pd.to_datetime(["2021-03-01","2021-06-01","2021-09-01"]))]
w21 = pre.groupby(["ent","provincia"], observed=True).depositos_prov.mean().reset_index()
w21 = w21[w21.depositos_prov > 0]
w21["w"] = w21.depositos_prov / w21.groupby("ent").depositos_prov.transform("sum")
p19 = pdp[pdp.ym.isin(pd.to_datetime(["2019-03-01","2019-06-01","2019-09-01","2019-12-01"]))]
w19 = p19.groupby(["ent","provincia"], observed=True).depositos_prov.mean().reset_index()
w19 = w19[w19.depositos_prov > 0]
w19["w"] = w19.depositos_prov / w19.groupby("ent").depositos_prov.transform("sum")
m = w21.merge(w19, on=["ent","provincia"], suffixes=("_21","_19"))
r = np.corrcoef(m.w_19, m.w_21)[0,1]
fig, ax = plt.subplots(figsize=(5.6, 5.6))
ax.scatter(m.w_19, m.w_21, s=14, color="navy", alpha=0.45, linewidths=0)
ax.plot([0,1],[0,1], color="black", lw=1)
ax.set_xlabel("Peso de la provincia en los depósitos del banco, 2019", fontsize=9)
ax.set_ylabel("Mismo peso, 2021 previo a las reformas", fontsize=9)
ax.text(0.03, 0.93, f"correlación = {r:.3f}", transform=ax.transAxes, fontsize=9)
ax.tick_params(labelsize=8); fig.tight_layout()
fig.savefig(os.path.join(figd, "av_pesos_estabilidad.png"), dpi=170); plt.close(fig)

x = inc[inc.n_pers_min1_cta.notna()].copy()
x["adop"] = 100 * x.n_pers_min1_cta_pago / x.n_pers_min1_cta
x = x.sort_values(["provincia","ym"])
x["d"] = x.groupby("provincia").adop.diff()
x["d_dif"] = x.d - x.groupby("ym").d.transform("mean")
sd_all = x.groupby("ym").d_dif.std()
xs = x[x.provincia != "SANTA FE"].copy()
xs["d_dif"] = xs.d - xs.groupby("ym").d.transform("mean")
sd_sin = xs.groupby("ym").d_dif.std()
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(sd_all.index, sd_all.values, color="navy", lw=1.7, label="Todas las provincias")
ax.plot(sd_sin.index, sd_sin.values, color="maroon", lw=1.7, label="Sin Santa Fe")
for d, lbl in [(QR, QR_LBL), (PULL, PULL_LBL)]:
    ax.axvline(d, color="black", lw=1)
    ax.annotate(lbl, xy=(d, ax.get_ylim()[1]*0.97), xytext=(5, -2), textcoords="offset points",
                fontsize=8, va="top", ha="left")
ax.annotate("Billetera Santa Fe:\ncaída de 24 pp en un trimestre", xy=(pd.Timestamp("2022-03-01"), sd_all.max()),
            xytext=(30, -12), textcoords="offset points", fontsize=8, color="navy")
ax.set_ylabel("Desvío estándar entre provincias del crecimiento\ntrimestral de adopción, neto del nacional (pp)", fontsize=9)
ax.legend(fontsize=8, frameon=False, loc="center right"); ax.tick_params(labelsize=8)
fig.tight_layout(); fig.savefig(os.path.join(figd, "av_sd_crecimiento.png"), dpi=170); plt.close(fig)
print("corr pesos:", round(r,3))

## 3. La serie del sistema: fondeo a la vista remunerado de personas jurídicas (311724) y la escalera de encajes

In [ ]:
sis = bancos.groupby("ym")[["dep_ctaesp_cint","dep_spnf"]].sum()
sis["share"] = 100 * sis.dep_ctaesp_cint / sis.dep_spnf
sis = sis.loc["2019-01-01":]
fig, ax = plt.subplots(figsize=(10.5, 4.6))
ax.plot(sis.index, sis.share, color="navy", lw=1.9)
ladder = [("2020-04-01", "A 6992: se crea la categoría\nencaje 0%"),
          ("2022-01-01", "A 7429: encaje 100%\na la custodia de PSP"),
          ("2024-04-01", "A 7988: 10%"), ("2024-05-01", "A 8000: 15%"),
          ("2024-11-01", "A 8119: 20%"), ("2025-08-01", "A 8281/8289: 40%"),
          ("2025-09-01", "A 8306: 43,5%"), ("2025-12-01", "A 8355: 40%")]
tops = [0.98, 0.72, 0.98, 0.55, 0.83, 0.98, 0.60, 0.83]
for (d, lbl), tp in zip(ladder, tops):
    d = pd.Timestamp(d)
    style = "--" if "7429" in lbl or "6992" in lbl else "-"
    ax.axvline(d, color="black", lw=0.9, ls=style)
    ax.annotate(lbl, xy=(d, ax.get_ylim()[0] + tp*(ax.get_ylim()[1]-ax.get_ylim()[0])),
                xytext=(4, 0), textcoords="offset points", fontsize=7.2, va="top")
ax.set_ylabel("Cta cte especial de personas jurídicas con interés (311724),\n% de los depósitos SPNF en pesos del sistema", fontsize=9)
ax.tick_params(labelsize=8)
ax.text(0.01, 0.02, "Líneas llenas: encaje sobre el haber a la vista de FCI money market. Punteadas: creación de la categoría y encaje a la custodia de PSP.",
        transform=ax.transAxes, fontsize=7.5, color="0.35")
fig.tight_layout(); fig.savefig(os.path.join(figd, "av_serie_311724.png"), dpi=170); plt.close(fig)
print(sis.share.loc[["2019-01-01","2021-10-01","2024-01-01","2024-10-01","2026-04-01"]].round(1).to_string())

## 4. Primera pasada: medidas de depósitos por exposición, bancos minoristas

Muestra: bancos con vista minorista mayor al 10% de sus depósitos SPNF en el pre. Exposición: adopción provincial ponderada por los depósitos del banco, excluyendo CABA de los pesos, fijada al pre. Grupos: tercil alto contra tercil bajo. Índice 100 en el mes previo a cada reforma.

In [ ]:
adop21 = inc[inc.ym == "2021-09-01"].copy()
adop21["adop"] = 100 * adop21.n_pers_min1_cta_pago / adop21.n_pers_min1_cta
adop21 = adop21.set_index("provincia").adop
wn = pre[pre.provincia != "CABA"].groupby(["ent","provincia"], observed=True).depositos_prov.mean().reset_index()
wn = wn[wn.depositos_prov > 0]
wn["w"] = wn.depositos_prov / wn.groupby("ent").depositos_prov.transform("sum")
wn["wA"] = wn.w * wn.provincia.map(adop21)
E = wn.groupby("ent").wA.sum().rename("E")
pre_b = bancos[(bancos.ym >= "2021-01-01") & (bancos.ym <= "2021-10-01")].groupby("ent")[["dep_vista_min","dep_spnf"]].sum()
retail = pre_b[pre_b.dep_vista_min / pre_b.dep_spnf > 0.10].index
b = bancos[bancos.ent.isin(retail)].merge(E, left_on="ent", right_index=True)
b["tercil"] = pd.qcut(b.E, 3, labels=[1,2,3]).astype(int) if False else b.E.map(lambda v: 0)
terc = pd.qcut(E[E.index.isin(retail)], 3, labels=[1,2,3]).astype(int)
b["tercil"] = b.ent.map(terc)
measures = [("dep_cajaahorro_r","Caja de ahorros"), ("dep_vista_min_r","Vista minorista (caja de ahorros + sueldo + CGU + UVA)"),
            ("dep_ctacte_r","Cuenta corriente sin interés"), ("dep_pf_total_r","Plazo fijo, todas las modalidades")]
shocks = [(QR, "QR interoperable, vigente 29-nov-2021"), (PULL, "Transferencias pull, vigente 30-sep-2022")]
fig, axes = plt.subplots(4, 2, figsize=(11, 10), sharex="col")
for j, (sh, collbl) in enumerate(shocks):
    base_m = sh - pd.DateOffset(months=1)
    win = b[(b.ym >= sh - pd.DateOffset(months=12)) & (b.ym <= sh + pd.DateOffset(months=12))]
    for i, (v, vlab) in enumerate(measures):
        ax = axes[i, j]
        g = win.groupby(["tercil","ym"], observed=True)[v].sum().reset_index()
        for t, col, lbl in [(1, "navy", "Exposición baja (tercil 1)"), (3, "maroon", "Exposición alta (tercil 3)")]:
            d = g[g.tercil == t].set_index("ym")[v]
            ax.plot(d.index, 100 * d / d.loc[base_m], color=col, lw=1.6, label=lbl)
        ax.axvline(sh, color="black", lw=1)
        if i == 0:
            ax.set_title(collbl, fontsize=10)
            ax.annotate("vigencia", xy=(sh, ax.get_ylim()[0]), xytext=(4, 6), textcoords="offset points", fontsize=7.5)
        ax.set_ylabel(vlab, fontsize=7.6); ax.tick_params(labelsize=7.5)
        if i == 0 and j == 0: ax.legend(fontsize=8, frameon=False)
fig.suptitle("Depósitos reales por grupo de exposición, bancos minoristas, índice 100 un mes antes de cada reforma", fontsize=10, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.985])
fig.savefig(os.path.join(figd, "av_grilla_minoristas.png"), dpi=160); plt.close(fig)
print("bancos minoristas:", len(retail), "| terciles:", terc.value_counts().to_dict())